In [5]:
# import libraries
import requests 
import pandas as pd
import io 
import os 
import numpy as np


In [6]:
# create a variable to store the file path
filepath_istat_data = '../raw/dati_istat.csv'

# create a logic to check if the file exists, if it does, read it into a dataframe, if not, fetch the data from the URL and save it to the file path
if os.path.exists(filepath_istat_data):
    df_istat_data = pd.read_csv(filepath_istat_data)
else:
    url='https://esploradati.istat.it/SDMXWS/rest/data/41_983'
    headers = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.content.decode('utf-8')
        df_istat_data = pd.read_csv(io.StringIO(data))
        df_istat_data.to_csv(filepath_istat_data, index=False)
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}")


In [7]:
# display the first 5 rows of the dataframe
df_istat_data.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
headers_comuni = {
    'Content-Type': 'application/json-patch+json',
    'Cookie': 'rxVisitor=...'  # copia il tuo cookie completo
}

body = {
    "orderFields": [],
    "orderDirects": [],
    "pFilterFields": [],
    "pFilterValues": []
}

for anno in range(2001, 2025):
    if os.path.exists(f'../raw/comuni_{anno}.csv'):
        df_comuni = pd.read_csv(f'../raw/comuni_{anno}.csv')
    else:
        url_comuni = f"https://situas.istat.it/ShibO2Module/api/Report/Spool/{anno}-12-31/74?&pdoctype=CSV"
        response_comuni = requests.post(url_comuni, headers=headers_comuni, json=body)
        if response_comuni.status_code == 200:
            data = response_comuni.content.decode('utf-8')
            df_comuni = pd.read_csv(io.StringIO(data))
            df_comuni.to_csv(f'../raw/comuni_{anno}.csv', index=False)
        else:
            print(f"Failed to fetch data for {anno}. Status code: {response_comuni.status_code}")
